In [4]:
import pandas as pd 
pd.read_csv("synthetic_autonomous_vehicle_data.csv")

,Time (s),Speed (km/h),Acceleration (m/s^2),Steering Angle (°),Distance to Obstacle (m),Brake Intensity (%)
0,0.0000,57.454012,0.355402,9.430654,84.914197,15.085803
1,0.1001,115.071431,-2.670689,-3.373060,105.249207,0.000000
2,0.2002,93.199394,0.760396,-0.359179,96.385580,3.614420
3,0.3003,79.865848,1.221171,-6.078046,90.223038,9.776962
4,0.4004,35.601864,1.119581,2.215601,72.043086,27.956914
...,...,...,...,...,...,...
995,99.5996,29.158207,-2.640045,-6.331156,58.111566,41.888434
996,99.6997,111.731358,-1.223538,7.099480,102.387204,0.000000
997,99.7998,33.681863,-0.074074,7.651198,62.250297,37.749703
998,99.8999,115.023735,-0.858604,-0.673805,113.709788,0.000000


# Convert the dataset siutable for classification

In [5]:
data = pd.read_csv("synthetic_autonomous_vehicle_data.csv")

data["Class"] = (data["Distance to Obstacle (m)"] <= 50).astype(int)
data.head()



,Time (s),Speed (km/h),Acceleration (m/s^2),Steering Angle (°),Distance to Obstacle (m),Brake Intensity (%),Class
0,0.0000,57.454012,0.355402,9.430654,84.914197,15.085803,0
1,0.1001,115.071431,-2.670689,-3.373060,105.249207,0.000000,0
2,0.2002,93.199394,0.760396,-0.359179,96.385580,3.614420,0
3,0.3003,79.865848,1.221171,-6.078046,90.223038,9.776962,0
4,0.4004,35.601864,1.119581,2.215601,72.043086,27.956914,0


# Classification Model

In [6]:
import numpy as np

X = data[["Speed (km/h)", "Acceleration (m/s^2)", "Steering Angle (°)"]].values
y = data["Class"].values

X = (X - X.mean(axis=0)) / X.std(axis=0)

np.random.seed(42)
indices = np.random.permutation(X.shape[0])
train_size = int(0.8 * len(indices))
X_train, X_test = X[indices[:train_size]], X[indices[train_size:]]
y_train, y_test = y[indices[:train_size]], y[indices[train_size:]]


#Build Model
class ClassificationModel:
    def __init__(self, learning_rate=0.01, num_iterations=1000):
        self.learning_rate = learning_rate
        self.num_iterations = num_iterations
        self.weights = None
        self.bias = None

    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-z))

    def train(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for i in range(self.num_iterations):
            linear_model = np.dot(X, self.weights) + self.bias

            y_predicted = self.sigmoid(linear_model)

            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)

            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

    def predict(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        y_predicted = self.sigmoid(linear_model)
        return (y_predicted >= 0.5).astype(int)


#Train the Model
model = ClassificationModel(learning_rate=0.01, num_iterations=1000)
model.train(X_train, y_train)

#Evaluate the Model
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Accuracy
train_accuracy = np.mean(y_pred_train == y_train)
test_accuracy = np.mean(y_pred_test == y_test)
print(f"Train Accuracy: {train_accuracy:.2f}")
print(f"Test Accuracy: {test_accuracy:.2f}")

# Confusion Matrix
def confusion_matrix(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    return np.array([[TP, FP], [FN, TN]])

cm = confusion_matrix(y_test, y_pred_test)
print("Confusion Matrix:")
print(cm)


Train Accuracy: 1.00
Test Accuracy: 1.00
Confusion Matrix:
[[  0   0]
 [  0 200]]
